<a href="https://colab.research.google.com/github/chloe8407/gpt-2.0-sherlock/blob/main/4_Sequence_Model_Sherlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. GPT-style Dataset + Minimal Sequence Model — Sherlock Holmes

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
if not Path("sherlock.txt").exists():
    !wget -q -O sherlock.txt {URL}

raw = open("sherlock.txt", "r", encoding="utf-8").read()
# Project Gutenberg 라이선스 헤더/푸터 제거 -> 본문만 학습
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
s = raw.find(start_marker)
e = raw.find(end_marker)
start = raw.find("\n", s) + 1 if s != -1 else 0
end = e if e != -1 else len(raw)
text = raw[start:end].strip()

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

In [2]:
print(text[:1000])

The Adventures of Sherlock Holmes

by Arthur Conan Doyle


Contents

   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the Blue Carbuncle
   VIII.  The Adventure of the Speckled Band
   IX.    The Adventure of the Engineer’s Thumb
   X.     The Adventure of the Noble Bachelor
   XI.    The Adventure of the Beryl Coronet
   XII.   The Adventure of the Copper Beeches




I. A SCANDAL IN BOHEMIA


I.

To Sherlock Holmes she is always _the_ woman. I have seldom heard him
mention her under any other name. In his eyes she eclipses and
predominates the whole of her sex. It was not that he felt any emotion
akin to love for Irene Adler. All emotions, and that one particularly,
were abhorrent to his cold, precise but admirably balanced mind. He
was, I take it, the most perfect reasoning and observing machine that
the worl

In [3]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 32])
yb.shape: torch.Size([64, 32])


In [4]:
xb[0]

tensor([ 6,  1, 30,  1, 51, 49, 62,  1, 67, 68, 49, 62, 52,  1, 68, 56, 57, 67,
         1, 67, 68, 66, 49, 57, 62,  1, 62, 63,  0, 60, 63, 62])

In [5]:
class TinySequenceLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)            # (B, T, C)
        pos = self.position_embedding(pos)[None] # (1, T, C)
        h = tok + pos
        logits = self.lm_head(h)                 # (B, T, V)
        return logits

model = TinySequenceLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 88])


In [6]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

print("initial loss:", sequence_cross_entropy(logits, yb).item())

initial loss: 4.736870288848877


In [7]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinySequenceLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 3.0201
epoch 10 | train loss 2.4263
epoch 20 | train loss 2.4237
epoch 30 | train loss 2.4256
epoch 40 | train loss 2.4199
epoch 50 | train loss 2.4228
epoch 60 | train loss 2.4244
epoch 70 | train loss 2.4219
epoch 80 | train loss 2.4237
epoch 90 | train loss 2.4234
epoch 99 | train loss 2.4232


In [8]:
@torch.no_grad()
def sample_sequence_model(model, block_size, stoi, itos, device, start_text="Sherlock Holmes", max_new_tokens=400):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_sequence_model(model, block_size, stoi, itos, device, start_text="Sherlock Holmes", max_new_tokens=400))

Sherlock Holmes methowha _, tsepalfoache “Gothate whan in
“Yer, ht, the sofug aied iens,
wid.”
u igalibe imy hasind od, It f tt

 CLouctr ghe
n whand I y p tre t in aveyone wienegacallf thiste. y kly, theriaks hicoures I teng p l rar ind wanled d t ca tade liseellsiemsave
pegawane thageanckeve thar omo
ad ownt. ll ther ass.” d as
tt in hanestry st tin bun tht Peithe s
ou f ulanlingedeachod, twadonout, lasomyomer
